<a href="https://colab.research.google.com/github/karthikpaii/workshop-MITE/blob/main/workshopday2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [106]:
pip install psycopg


In [107]:
import os
import sys
from typing import List, Literal, Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup
import psycopg
from psycopg.rows import dict_row

In [108]:
MOST_RUNS_TEST_URL = "https://www.bcci.tv/getStats?platform=international&type=men&s_type=batting&slug=batting_most_runs&format=test"
MOST_RUNS_ODI_URL = "https://www.bcci.tv/getStats?platform=international&type=men&s_type=batting&slug=batting_most_runs&format=odi"
TOP_WICKET_TAKERS_ODI_URL = "https://www.bcci.tv/getStats?platform=international&type=men&s_type=bowling&slug=bowling_top_wicket_takers&format=odi"
TOP_WICKET_TAKERS_TEST_URL = "https://www.bcci.tv/getStats?platform=international&type=men&s_type=bowling&slug=bowling_top_wicket_takers&format=test"

In [109]:
OUT_DIR = os.path.join(os.getcwd(), "out")

In [110]:
Discipline = Literal["batting", "bowling"]

jobs: list[tuple[str, str, Discipline]] = [
    (MOST_RUNS_TEST_URL, "bcci_test_most_runs", "batting"),
    (MOST_RUNS_ODI_URL, "bcci_odi_most_runs", "batting"),
    (TOP_WICKET_TAKERS_TEST_URL, "bcci_test_top_wickets", "bowling"),
    (TOP_WICKET_TAKERS_ODI_URL, "bcci_odi_top_wickets", "bowling"),
]

In [111]:
columns: dict[Discipline, list[str]] = {
"batting": [
        "Rank",
        "Name",
        "Matches",
        "Inns",
        "Avg",
        "SR",
        "HS",
        "Fours",
        "Sixes",
        "Fifties",
        "Centuries",
        "Runs",
    ],
    "bowling": [
        "Rank",
        "Name",
        "Matches",
        "Inns",
        "Avg",
        "Econ",
        "SR",
        "BBI",
        "Four_w",
        "Five_w",
        "Wkts",
    ],
}

In [112]:
saved_paths: List[str] = []

In [113]:
os.makedirs(OUT_DIR,exist_ok=True)

In [114]:
html_score:dict[str,str]=[]

In [115]:
import os
import requests


os.makedirs(OUT_DIR, exist_ok=True)

html_store: dict[str, str] = {}

for url, basename, kind in jobs:

        resp = requests.get(url, timeout=30)
        resp.raise_for_status()

        payload = resp.json()
        html = payload.get("html")

        if not html:
            print(f"HTML data found for {basename} does not exist.")
            continue


        html_store[basename] = html


        print(f"Got HTML data for {basename}")




Got HTML data for bcci_test_most_runs
Got HTML data for bcci_odi_most_runs
Got HTML data for bcci_test_top_wickets
Got HTML data for bcci_odi_top_wickets


In [116]:
def normalize_label(text: str) -> str:
    low = text.replace("\u2019", "'").replace(".", "").strip().lower()
    mapping = {
        "matches": "Matches",
        "inns": "Inns",
        "avg": "Avg",
        "sr": "SR",
        "hs": "HS",
        "runs": "Runs",
        "4's": "Fours",
        "4s": "Fours",
        "6's": "Sixes",
        "6s": "Sixes",
        "50's": "Fifties",
        "50s": "Fifties",
        "100's": "Centuries",
        "100s": "Centuries",
        "econ": "Econ",
        "economy": "Econ",
        "wkts": "Wkts",
        "wickets": "Wkts",
        "bbi": "BBI",
        "best bowling": "BBI",
        "best": "BBI",
        "4w": "Four_w",
        "5w": "Five_w",
    }
    return mapping.get(low, text)


def coerce_value(text: str):
    try:
        if "." in text:
            return float(text)
        return int(text)
    except Exception:
        return text.strip()
    print("Initial Helper function")

In [117]:
from bs4 import BeautifulSoup

def extract_data_from_html(html: str, kind: Discipline) -> tuple[list[dict], list[str]]:
    soup = BeautifulSoup(html, "lxml")

    table = soup.select_one(".stats-data-table-player table")
    if table is None:
        return [], columns[kind]

    records: list[dict] = []

    top = get_first_rank_player(soup)
    if top:
        records.append(top)

    for tr in table.select("tr"):
        tds = tr.find_all("td")
        if len(tds) < 3:
            continue

        sn_el = tds[0].find(["h5", "h6"]) or tds[0]
        name_el = tds[1].find("h6") or tds[1]
        try:
            sn = int((sn_el.get_text(strip=True) or "0").strip())
        except Exception:
            sn = None

        name = name_el.get_text(strip=True)
        row: dict = {"Rank": sn, "Name": name}

        for td in tds[2:]:
            val_el = td.find("h6")
            lab_el = td.find("span")
            if not val_el or not lab_el:
                continue

            val_txt = val_el.get_text(strip=True)
            lab_txt = lab_el.get_text(strip=True)
            key = normalize_label(lab_txt)
            row[key] = coerce_value(val_txt)

        records.append(row)

    for r in records:
        for c in columns[kind]:
            if c not in r:
                r[c] = None

    return records, columns[kind]


In [118]:
def get_first_rank_player(soup: BeautifulSoup):
    name_container = soup.select_one(".player-name-trw")
    stat_table = soup.select_one(".ranking-top-table table")

    if not name_container or not stat_table:
        print("Top-ranked player section not found")
        return None

    name = " ".join(name_container.stripped_strings)

    stats = {"Rank": 1, "Name": name}
    for td in stat_table.select("td"):
        label = td.find("span").get_text(strip=True)
        value = td.find("p").get_text(strip=True)
        key = normalize_label(label)
        stats[key] = coerce_value(value)

    return stats


In [119]:
from IPython.display import display
import os
import pandas as pd

# Create the output directory if it doesn't exist
os.makedirs(OUT_DIR, exist_ok=True)
saved_paths = []
for url, basename, discipline in jobs:
   # TODO: get the HTML string you stored earlier for this basename (from html_store)


    html = html_store.get(basename)

    records, cols = extract_data_from_html(html, discipline)

    if not records:
        print(f"No records found for {basename}!")
        continue

    df = pd.DataFrame(records, columns=cols)
    # TODO: build a DataFrame using the correct columns list for this discipline
    print(f"--------{basename}----------")
    display(df)
    print("\n\n\n")
    # TODO: build the CSV path for this basename inside OUT_DIR
    # (example: use os.path.join with OUT_DIR and basename)





    path = os.path.join(OUT_DIR, f"{basename}.csv")
    df.to_csv(path, index=False)

    saved_paths.append(path)


--------bcci_test_most_runs----------


,Rank,Name,Matches,Inns,Avg,SR,HS,Fours,Sixes,Fifties,Centuries,Runs
0,1,Sachin Tendulkar,200,329,53.78,54.09,248,2058,69,68,51,15921
1,2,Rahul Dravid,164,286,52.31,42.51,270,1655,21,63,36,13288
2,3,Sunil Gavaskar,125,214,51.12,66.04,236,1016,26,45,34,10122
3,4,Virat Kohli,123,210,46.85,55.57,254,1027,30,31,30,9230
4,5,VVS Laxman,134,225,45.97,49.37,281,1135,5,56,17,8781
...,...,...,...,...,...,...,...,...,...,...,...,...
187,188,Raju Kulkarni,3,2,1.00,-,2,-,-,-,-,2
188,189,Vijay Dahiya,2,1,0.00,40.0,2,-,-,-,-,2
189,190,Rahul Sanghvi,1,2,1.00,9.09,2,-,-,-,-,2
190,191,Shahbaz Nadeem,2,3,0.50,3.33,1,-,-,-,-,1






--------bcci_odi_most_runs----------


,Rank,Name,Matches,Inns,Avg,SR,HS,Fours,Sixes,Fifties,Centuries,Runs
0,1,Sachin Tendulkar,463,452,44.83,86.23,200,2016,195,96,49,18426
1,2,Virat Kohli,311,299,58.71,93.82,183,1376,168,77,54,14797
2,3,Rohit Sharma,282,274,48.84,92.74,264,1090,357,61,33,11577
3,4,Sourav Ganguly,311,300,41.02,73.70,183,1122,190,72,22,11363
4,5,Rahul Dravid,344,318,39.16,71.23,153,950,42,83,12,10889
...,...,...,...,...,...,...,...,...,...,...,...,...
232,233,Yograj Singh,6,4,0.50,8.33,1,-,-,-,-,1
233,234,Rahul Sharma,4,1,1.00,50.00,1,-,-,-,-,1
234,235,Sudeep Tyagi,4,1,0.00,50.00,1,-,-,-,-,1
235,236,Siddarth Kaul,3,2,0.50,33.33,1,-,-,-,-,1






--------bcci_test_top_wickets----------


,Rank,Name,Matches,Inns,Avg,Econ,SR,BBI,Four_w,Five_w,Wkts
0,1,Anil Kumble,132,236,29.65,2.69,65.9,None,None,None,619
1,2,Ravichandran Ashwin,106,200,24.00,NaN,50.7,None,None,None,537
2,3,Kapil Dev,131,227,29.64,NaN,63.9,None,None,None,434
3,4,Harbhajan Singh,103,190,32.46,NaN,68.5,None,None,None,417
4,5,Ravindra Jadeja,89,167,25.11,NaN,58.1,None,None,None,348
...,...,...,...,...,...,...,...,...,...,...,...
176,177,Yograj Singh,1,1,63.00,NaN,90.0,None,None,None,1
177,178,Pankaj Roy,43,7,66.00,NaN,104.0,None,None,None,1
178,179,Vinay Kumar,1,1,73.00,NaN,78.0,None,None,None,1
179,180,Nari Contractor,31,10,80.00,NaN,186.0,None,None,None,1






--------bcci_odi_top_wickets----------


,Rank,Name,Matches,Inns,Avg,Econ,SR,BBI,Four_w,Five_w,Wkts
0,1,Anil Kumble,271,265,30.89,4.3,43.0,None,None,None,337
1,2,Javagal Srinath,229,227,28.08,NaN,37.8,None,None,None,315
2,3,Ajit Agarkar,191,188,27.85,NaN,32.9,None,None,None,288
3,4,Zaheer Khan,200,197,29.43,NaN,35.8,None,None,None,282
4,5,Harbhajan Singh,236,227,33.35,NaN,46.3,None,None,None,269
...,...,...,...,...,...,...,...,...,...,...,...
163,164,Ravi Bishnoi,1,1,69.00,NaN,48.0,None,None,None,1
164,165,Vinod Kambli,104,1,7.00,NaN,4.0,None,None,None,1
165,166,Rohan Gavaskar,11,2,74.00,NaN,72.0,None,None,None,1
166,167,Vivek Razdan,3,3,77.00,NaN,84.0,None,None,None,1
